# LFW 04. pgvector candidate search, calibration, and certification

> **PCA sweep update**: evaluates Origin-512 and PCA-448/384/256/128 using exact and HNSW search. Estimated runtime is 40-160 minutes. Progress is reported per profile and every 100 probes. If interrupted, restart the kernel and run from the first cell.

## 예상 소요 시간

| 실행 모드 | 예상 시간 | 주요 작업 |
| --- | ---: | --- |
| `EXECUTE_STAGE=False` | 1초 미만 | run과 이전 phase 확인 준비 |
| `EXECUTE_STAGE=True` | 약 10~40분 | calibration exact SQL, test exact/HNSW SQL, certificate, baseline 측정 |

> 100개 probe마다 진행률과 30초 heartbeat를 출력합니다.

목표: PCA-256 pgvector HNSW Top-K 후보를 검색하고, candidate recall을 측정한 뒤 원본 query 512D와 reconstructed template으로 certificate를 계산합니다. 임계값은 calibration split의 목표 FPIR에서 origin-512와 PCA-256 점수 공간별로 고정합니다.

> **재시작 규칙**: Kernel Restart 후 Run All을 사용합니다. 03까지 완료된 새 run에서 실행하십시오. 완료 run은 수정하지 않습니다.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate.resolve()
    raise RuntimeError('Run Jupyter from the ronbun repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
from research.runtime import ProgressReporter, RunStore, resolve_active_run

EXECUTE_STAGE = True
RUN_ROOT = PROJECT_ROOT / 'runs' / 'lfw'
LEGACY_RUN_ROOT = PROJECT_ROOT / 'runs'
try:
    RUN_DIR = resolve_active_run(RUN_ROOT)
except FileNotFoundError:
    RUN_ROOT = LEGACY_RUN_ROOT
    RUN_DIR = resolve_active_run(RUN_ROOT)
PROGRESS = ProgressReporter('04 pgvector search/calibration/certification', heartbeat_seconds=30)


## Plan

- Calibration scope에서 origin-512와 PCA-256 pgvector exact 검색으로 점수 공간별 목표 FPIR 임계값을 선택합니다.
- Test probe마다 origin-512 exact, PCA-256 exact, PCA-256 HNSW Top-K를 각각 측정합니다.
- HNSW candidate recall, 압축 rank inversion, threshold crossing을 분리합니다.
- Certificate는 원본 query를 사용하므로 query angular error를 0으로 고정합니다.
- Defer만 origin-512 exact fallback을 사용한 것으로 시뮬레이션합니다.

In [ ]:
def attach_run(run_dir: Path) -> tuple[RunStore, dict]:
    run = RunStore.open(run_dir)
    manifest = json.loads(run.manifest_path.read_text(encoding='utf-8'))
    if manifest.get('status') == 'completed' or (run_dir / 'COMPLETED').exists():
        raise RuntimeError('Completed runs are immutable.')
    return run, manifest


def latest_completed_attempt(run_dir: Path, phase_name: str) -> int:
    attempts_dir = run_dir / 'phases' / phase_name / 'attempts'
    completed = []
    for manifest_path in sorted(attempts_dir.glob('A*/phase_manifest.json')):
        payload = json.loads(manifest_path.read_text(encoding='utf-8'))
        if payload.get('status') == 'completed':
            completed.append(int(payload['attempt']))
    if not completed:
        raise RuntimeError(f'No completed attempt for {phase_name}')
    return max(completed)


preflight = {
    'execute_stage': EXECUTE_STAGE,
    'run_dir_resolved': str(RUN_DIR),
    'run_manifest_exists': bool(RUN_DIR and (RUN_DIR / 'run_manifest.json').is_file()),
    'search_backend': 'postgresql_pgvector_exact_and_hnsw',
}
preflight

## Execute and record

HNSW 후보 집합에 대한 certificate는 전역 보증으로 표현하지 않습니다. `certification_global_claim=False`와 candidate recall을 함께 기록합니다.

In [ ]:
result = {'status': 'not_executed', **preflight}
if EXECUTE_STAGE:
    import pandas as pd
    from research.compression import ORIGIN_512, PCACompressor, PCA_256
    from research.database import create_database_engine, load_database_settings
    from research.experiments import (
        LFWTemplateScope,
        build_lfw_certification_inputs,
        calibrate_lfw_pgvector_threshold,
        run_lfw_pgvector_search,
        write_vector_frame_csv,
    )

    with PROGRESS.step('run/input 및 00·02·03 artifact 검증', expected='10초 미만'):
        run, run_manifest = attach_run(RUN_DIR)
        run.verify_inputs()
        for upstream in (
            '00_protocol_and_run_freeze',
            '02_compressor_fit',
            '03_compressed_materialization_and_index',
        ):
            run.verify_phase_artifacts(upstream)
        config = run_manifest['config']
        protocol_attempt = latest_completed_attempt(RUN_DIR, '00_protocol_and_run_freeze')
        protocol_suffix = f'A{protocol_attempt:03d}'
        protocol_dir = RUN_DIR / 'artifacts' / '00_protocol_and_run_freeze'
        test_protocol_frames = {
            role: pd.read_csv(protocol_dir / f'{role}_{protocol_suffix}.csv')
            for role in ('gallery', 'registered_probes', 'known_unknown_probes', 'unknown_unknown_probes')
        }
        compressor_attempt = latest_completed_attempt(RUN_DIR, '02_compressor_fit')
        compressor_suffix = f'A{compressor_attempt:03d}'
        pca_dimensions = [int(value) for value in config['compression']['pca'].get('dimensions', [448, 384, 256, 128])]
        pca_paths = {
            f'pca_{dimension}': RUN_DIR / 'artifacts' / '02_compressor_fit' / f'pca_{dimension}_{compressor_suffix}.joblib'
            for dimension in pca_dimensions
        }
        pcas = {profile: PCACompressor.load(path) for profile, path in pca_paths.items()}
        materialization_attempt = latest_completed_attempt(
            RUN_DIR, '03_compressed_materialization_and_index'
        )
        materialization_suffix = f'A{materialization_attempt:03d}'
        materialization_dir = RUN_DIR / 'artifacts' / '03_compressed_materialization_and_index'
        materialization_summary = json.loads(
            (materialization_dir / f'materialization_summary_{materialization_suffix}.json').read_text(encoding='utf-8')
        )
        calibration_protocol_frames = {
            role: pd.read_csv(materialization_dir / f'calibration_{role}_{materialization_suffix}.csv')
            for role in ('gallery', 'registered_probes', 'known_unknown_probes', 'unknown_unknown_probes')
        }
        gallery_counts = test_protocol_frames['gallery'].groupby('identity_id').size()
        if gallery_counts.nunique() != 1:
            raise ValueError('LFW test gallery enrollment count must be uniform.')
        enrollment_target = int(gallery_counts.iloc[0])
        engine = create_database_engine(load_database_settings())
        pca_model_uids = {
            str(profile): str(uid)
            for profile, uid in materialization_summary['pca_model_uids'].items()
        }
        search_config = config['search']
        candidate_k = int(search_config.get('candidate_k', 10))
        ef_search = int(search_config.get('hnsw_ef_search', 40))
        target_fpir = float(config['calibration']['target_fpir'])

    with run.phase('04_probe_search_and_certification') as phase:
        suffix = f'A{phase.attempt:03d}'

        def report(message: str, details: dict[str, object]) -> None:
            PROGRESS.emit(message, **details)

        with PROGRESS.step('origin/PCA calibration exact SQL 및 threshold 선택', expected='5~20분'):
            calibration_feature_frames = []
            search_feature_frames = []
            calibration_summary = {}
            search_summary = {}
            thresholds = {}
            test_protocol_coverage = {}
            calibration_protocol_coverage = {}
            for profile, compressor in pcas.items():
                test_bundle = build_lfw_certification_inputs(
                    engine, run_uid=run.run_id, protocol_frames=test_protocol_frames,
                    project_root=PROJECT_ROOT, compression_profile=profile, pca=compressor,
                )
                calibration_bundle = build_lfw_certification_inputs(
                    engine, run_uid=run.run_id, protocol_frames=calibration_protocol_frames,
                    project_root=PROJECT_ROOT, compression_profile=profile, pca=compressor,
                    allow_empty_unknown_unknown=True,
                )
                test_scope = LFWTemplateScope(
                    run_uid=run.run_id, protocol_name='lfw_test',
                    model_uid=pca_model_uids[profile], enrollment_target=enrollment_target,
                )
                calibration_scope = LFWTemplateScope(
                    run_uid=run.run_id, protocol_name='lfw_calibration',
                    model_uid=pca_model_uids[profile], enrollment_target=enrollment_target,
                )
                origin_threshold, origin_features, origin_summary = calibrate_lfw_pgvector_threshold(
                    engine, probes=calibration_bundle.probes, scope=calibration_scope,
                    target_fpir=target_fpir, compression_profile=ORIGIN_512, progress=report,
                )
                pca_threshold, pca_features, pca_summary = calibrate_lfw_pgvector_threshold(
                    engine, probes=calibration_bundle.probes, scope=calibration_scope,
                    target_fpir=target_fpir, compression_profile=profile, progress=report,
                )
                origin_features['sweep_profile'] = profile
                pca_features['sweep_profile'] = profile
                calibration_feature_frames.extend([origin_features, pca_features])
                calibration_summary[profile] = {ORIGIN_512: origin_summary, profile: pca_summary}
                thresholds[profile] = {ORIGIN_512: origin_threshold, profile: pca_threshold}
                profile_features, profile_search_summary = run_lfw_pgvector_search(
                    engine, probes=test_bundle.probes, templates=test_bundle.templates,
                    scope=test_scope, origin_threshold=origin_threshold,
                    pca_threshold=pca_threshold, candidate_k=candidate_k,
                    ef_search=ef_search, compression_profile=profile, progress=report,
                )
                search_feature_frames.append(profile_features)
                search_summary[profile] = profile_search_summary
                test_protocol_coverage[profile] = test_bundle.coverage
                calibration_protocol_coverage[profile] = calibration_bundle.coverage
            calibration_features = pd.concat(calibration_feature_frames, ignore_index=True)
            features = pd.concat(search_feature_frames, ignore_index=True)
        PROGRESS.emit(
            'profile별 calibration threshold 고정',
            origin_threshold=origin_threshold,
            pca_threshold=pca_threshold,
            target_fpir=target_fpir,
        )
        with PROGRESS.step('test exact/HNSW candidate 검색 및 certificate', expected='5~30분'):
            PROGRESS.emit('PCA sweep search already completed', profiles=list(search_summary))

        calibration_features_path = phase.attempt_dir / f'calibration_features_{suffix}.csv'
        calibration_summary_path = phase.attempt_dir / f'calibration_summary_{suffix}.json'
        features_path = phase.attempt_dir / f'certified_features_{suffix}.csv'
        summary_path = phase.attempt_dir / f'certification_summary_{suffix}.json'
        calibration_features.to_csv(
            calibration_features_path, index=False, encoding='utf-8', lineterminator='\n'
        )
        calibration_summary_path.write_text(
            json.dumps(calibration_summary, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        write_vector_frame_csv(features, features_path)
        combined_summary = {
            'input_mode': 'pgvector_template_search',
            'compression_profiles': list(pcas),
            'candidate_scope': 'candidate_set',
            'certificate_query_space': 'origin_512',
            'certificate_template_space': 'profile_specific_pca_reconstructed_512',
            'calibration': calibration_summary,
            'search': search_summary,
            'thresholds': thresholds,
            'materialization_storage': {
                ORIGIN_512: materialization_summary['storage_bytes']['embedding_512'],
                'pca_256': materialization_summary['storage_bytes']['embedding_256'],
                **materialization_summary['pca_sweep']['storage_bytes'],
            },
            'test_protocol_coverage': test_protocol_coverage,
            'calibration_protocol_coverage': calibration_protocol_coverage,
        }
        summary_path.write_text(
            json.dumps(combined_summary, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        for artifact in (
            calibration_features_path,
            calibration_summary_path,
            features_path,
            summary_path,
        ):
            phase.publish_artifact(artifact)
        phase.record_counts(
            calibration_probes=len(calibration_features),
            test_probes=len(features),
            gallery_templates=sum(int(value['certification_gallery_size'].iloc[0]) for value in search_feature_frames),
            candidate_k=candidate_k,
        )
        phase.record(
            'pgvector_search_contract',
            thresholds=thresholds,
            threshold_source='calibration_profile_specific_target_fpir',
            candidate_scope='candidate_set',
            global_certificate_claim=False,
            query_angular_error=0.0,
            hnsw_ef_search=ef_search,
        )
    PROGRESS.emit(
        '04 완료',
        origin_threshold=origin_threshold,
        pca_threshold=pca_threshold,
        candidate_recall={profile: value['candidate_contains_origin_top1_rate'] for profile, value in search_summary.items()},
        fallback_rate={profile: value['certification']['exact_fallback_rate'] for profile, value in search_summary.items()},
    )
    result = {
        'status': 'completed',
        'run_id': run.run_id,
        'thresholds': thresholds,
        'calibration': calibration_summary,
        'search': search_summary,
    }
else:
    PROGRESS.emit('검토 모드 완료: calibration/DB 검색/certificate를 실행하지 않음', expected='1초 미만')
result

## Final check

다음을 함께 확인합니다.

- calibration `target_fpir`, 실제 FPIR, 선택 threshold
- HNSW candidate가 origin exact top-1과 true identity를 포함하는 비율
- PCA exact/HNSW rank inversion과 threshold crossing
- `certified_query_angular_error == 0`
- candidate-scoped certificate coverage와 origin exact fallback 비율
- exact/HNSW/system latency P50/P95